# Partición por conglomerado: entrenamiento, prueba y pliegues

**Entrada:** `dataset_final.csv` (cuaderno 03). **Salida:** `data/processed/particion.csv` (una fila por hogar con su conjunto y su pliegue). No modifica el dataset: solo registra a qué conjunto y pliegue pertenece cada hogar.

## ¿Por qué se parte por conglomerado y no hogar por hogar?
En la ENAHO, los hogares se eligen por **conglomerados** (zonas de unas decenas de viviendas). Los hogares de una misma zona se parecen mucho entre sí: mismo tipo de calle, mismos servicios, misma situación económica. Si un hogar de una zona cae en entrenamiento y su vecino en prueba, el modelo "reconoce" a la zona y el resultado parece mejor de lo que sería con hogares de zonas nuevas. Para evitarlo, **todos los hogares de un conglomerado van siempre juntos**.

## Cómo se hace
1. **Prueba (≈ 20 %):** se reserva una quinta parte de los conglomerados. Se usa **una sola vez**, al final, para la evaluación.
2. **Entrenamiento (≈ 80 %):** se divide en **5 pliegues**, también por conglomerado. Sirven para ajustar hiperparámetros y elegir umbrales (se entrena con 4 pliegues y se valida en el que queda fuera, 5 veces).
3. En ambos pasos se **estratifica por pobreza y área** (pobre/no pobre × urbano/rural): cada parte conserva una proporción parecida de hogares pobres (≈ 18 %) y de hogares urbanos, porque con una clase minoritaria un reparto al azar podría dejar muy pocos pobres en alguna parte, y porque más adelante se revisan los errores por área.

Se usa `StratifiedGroupKFold` de scikit-learn, que hace justo esto: grupos completos y proporción de clases parecida. La semilla queda fija para que el resultado se pueda repetir.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

RAIZ = Path.cwd().resolve().parent
PROCESSED = RAIZ / "data" / "processed"

SEMILLA = 42
N_PARTES_PRUEBA = 5    # 1 de 5 partes = 20 % de los conglomerados va a prueba
K_PLIEGUES = 5         # pliegues de validación cruzada dentro del entrenamiento


base = pd.read_csv(PROCESSED / "dataset_final.csv", encoding="utf-8-sig",
                   dtype={"id_hogar": str, "conglome": str, "ubigeo": str, "departamento": str})
print(f"{len(base):,} hogares")

## Paso 1. Mirar los conglomerados

**Por qué:** la partición reparte conglomerados, no hogares. Conviene saber cuántos hay y qué tan distintos son entre sí: si algunos tuvieran muchísimos hogares, los pliegues quedarían desiguales.

In [ ]:
cong = base.groupby("conglome").agg(hogares=("id_hogar", "size"), pobres=("y", "sum"))
print(f"Conglomerados: {len(cong):,}")
print("Hogares por conglomerado:")
print(cong["hogares"].describe().round(1).to_string())
print(f"\nConglomerados sin ningún hogar pobre: {(cong['pobres'] == 0).mean():.1%}")

## Paso 2. Separar entrenamiento y prueba

**Por qué:** la prueba debe ser datos que el modelo nunca vio, ni siquiera zonas parecidas. `StratifiedGroupKFold` con 5 partes devuelve, en cada vuelta, un reparto 80/20; tomamos la primera vuelta como el reparto definitivo.

Se verifica que **ningún conglomerado quede en los dos conjuntos** (si pasara, el cuaderno se detiene) y que la proporción de pobres sea parecida.

In [ ]:
base["_estrato"] = base["y"].astype(str) + "_" + base["area_urbana"].astype(str)   # pobreza x área

sgkf = StratifiedGroupKFold(n_splits=N_PARTES_PRUEBA, shuffle=True, random_state=SEMILLA)
idx_ent, idx_prueba = next(sgkf.split(base, base["_estrato"], groups=base["conglome"]))

base["conjunto"] = "entrenamiento"
base.iloc[idx_prueba, base.columns.get_loc("conjunto")] = "prueba"

c_ent = set(base.loc[base["conjunto"] == "entrenamiento", "conglome"])
c_pru = set(base.loc[base["conjunto"] == "prueba", "conglome"])
assert not (c_ent & c_pru), "hay conglomerados en entrenamiento y en prueba"
print("Ningún conglomerado está en ambos conjuntos: OK")

resumen = base.groupby("conjunto").agg(hogares=("id_hogar", "size"), conglomerados=("conglome", "nunique"),
                                       pobres=("y", "sum"), pct_pobres=("y", "mean"))
resumen["pct_pobres"] = (resumen["pct_pobres"] * 100).round(2)
resumen["pct_hogares"] = (resumen["hogares"] / len(base) * 100).round(1)
print(resumen.to_string())

## Paso 3. Pliegues dentro del entrenamiento

**Por qué:** para elegir hiperparámetros (profundidad del árbol, K del KNN, regularización, capas del MLP) hace falta validar muchas veces sin tocar la prueba. Se divide el entrenamiento en 5 pliegues, cada uno con conglomerados completos y proporción de pobres parecida. Cada hogar queda en **un solo** pliegue; la columna `pliegue` toma valores de 0 a 4 (y -1 para los hogares de prueba, que no tienen pliegue).

In [ ]:
ent = base[base["conjunto"] == "entrenamiento"].reset_index(drop=True)
sgkf2 = StratifiedGroupKFold(n_splits=K_PLIEGUES, shuffle=True, random_state=SEMILLA)
ent["pliegue"] = -1
for k, (_, idx_val) in enumerate(sgkf2.split(ent, ent["_estrato"], groups=ent["conglome"])):
    ent.loc[idx_val, "pliegue"] = k

assert (ent["pliegue"] >= 0).all(), "algún hogar quedó sin pliegue"
assert (ent.groupby("conglome")["pliegue"].nunique() == 1).all(), "un conglomerado quedó repartido entre pliegues"
print("Cada conglomerado está en un solo pliegue: OK")

tabla = ent.groupby("pliegue").agg(hogares=("id_hogar", "size"), conglomerados=("conglome", "nunique"),
                                   pobres=("y", "sum"), pct_pobres=("y", "mean"))
tabla["pct_pobres"] = (tabla["pct_pobres"] * 100).round(2)
print(tabla.to_string())

## Paso 4. Comprobar que la prueba se parece al entrenamiento

**Por qué:** la proporción de pobres ya se controló, pero si todos los hogares de una región cayeran en prueba, la evaluación no representaría al país. Se compara la composición por área (urbana/rural) y por departamento; las diferencias deberían ser de uno o dos puntos porcentuales, no de decenas.

In [ ]:
ref = base.copy()
print("% de hogares urbanos:")
print((ref.groupby("conjunto")["area_urbana"].mean() * 100).round(1).to_string())

dep = pd.crosstab(ref["departamento"], ref["conjunto"], normalize="columns") * 100
dep["diferencia"] = (dep["prueba"] - dep["entrenamiento"]).abs()
print(f"\nDepartamentos: mayor diferencia de peso entre prueba y entrenamiento = {dep['diferencia'].max():.2f} puntos porcentuales")
print(dep.sort_values("diferencia", ascending=False).head(5).round(2).to_string())

## Paso 5. Guardar la partición

**Por qué:** el cuaderno de modelos debe usar **exactamente** el mismo reparto. Se guarda una tabla con `id_hogar`, `conglome`, `conjunto` y `pliegue`; no se guardan copias de los datos, solo la asignación.

In [ ]:
pliegue_por_hogar = ent.set_index("id_hogar")["pliegue"]
particion = base[["id_hogar", "conglome", "conjunto"]].copy()
particion["pliegue"] = particion["id_hogar"].map(pliegue_por_hogar).fillna(-1).astype(int)

assert particion["id_hogar"].is_unique and len(particion) == len(base)
particion.to_csv(PROCESSED / "particion.csv", index=False, encoding="utf-8-sig")
print("Guardada:", PROCESSED / "particion.csv")
print(particion.groupby(["conjunto", "pliegue"]).size().to_string())